# Notebook 5 — Self-Quiz

**20 questions covering everything from N1 to N4.** Mix of multiple-choice and predict-the-state. Each question is auto-checked. Final score at the end.

## How this works

Each question has two cells:
1. A **predict cell** where you assign your answer to a variable.
2. A **check cell** that tells you right/wrong with a brief explanation.

**Don't run the check cell until you've made a real attempt.** A blank prediction (leaving `...`) counts as wrong.

## Sections

- **A** — Syntax & Grammar (Q1–4)
- **B** — States and updates (Q5)
- **C** — Evaluators `A` and `B` (Q6–7)
- **D** — Small-step rules (Q8–11)
- **E** — Tracing programs (Q12–14, Q19–20)
- **F** — Reasoning about the semantics (Q15–18)

Run all cells **in order** — the score tracker is a stateful global.

In [1]:
import sys, json, base64
from pathlib import Path

# Make this work whether jupyter was started from notebooks/ or from the repo root.
for candidate in [Path.cwd(), Path.cwd() / 'notebooks', Path.cwd().parent / 'notebooks']:
    if (candidate / 'while_lang.py').exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise ImportError("could not find while_lang.py — start jupyter from the notebooks/ folder")

from while_lang import parse, run, trace, A, B, step, step_iter, Config, StepBudgetExceeded

# Score tracker.
_RESULTS: dict[int, bool] = {}

# Answer key, base64-encoded so the answers aren't lying around in plaintext.
# You CAN decode it if you really want to — but you'd be cheating yourself out of the grok.
_KEY_B64 = (
    "eyIxIjogeyJhIjogIkIiLCAiZSI6ICJ0dCBpcyBhIGJvb2xlYW4gZXhwcmVzc2lvbiwgbm90IGFuIGFleHAuXG54ID0gMCBpcyBh"
    "IGJvb2xlYW4gZXhwcmVzc2lvbiAodGhlIGVxdWFsaXR5IG9wZXJhdG9yKS5cbmlmLXRoZW4tZWxzZSBpcyBhIHN0YXRlbWVudCwg"
    "bm90IGFuIGV4cHJlc3Npb24uXG5Pbmx5IHggKyAxIG1hdGNoZXMgPGFleHA+IDo6PSA8YWV4cD4gKyA8YWV4cD4uIn0sICIyIjog"
    "eyJhIjogIkMiLCAiZSI6ICJ4ICsgMSBpcyBhbiBhcml0aG1ldGljIGV4cHJlc3Npb24sIG5vdCBhIGJvb2xlYW4uIFRoZSBiZXhw"
    "IGdyYW1tYXIgaGFzIG5vIFwiK1wiIHJ1bGUuIn0sICIzIjogeyJhIjogIkMiLCAiZSI6ICJEZSBNb3JnYW46IGEg4oioIGIg4omh"
    "IMKsKMKsYSDiiKcgwqxiKS5cblRoZXJlIGlzIG5vIHwgb3IgKyBvcGVyYXRvciBvbiBib29sZWFucyBpbiBXaGlsZS5cbiEoYSAm"
    "IGIpIGlzIMKsKGEg4oinIGIpIOKAlCB0aGF0IGlzIE5BTkQsIG5vdCBPUi4ifSwgIjQiOiB7ImEiOiAiQiIsICJlIjogIlRoZSBz"
    "YW1lIHN0cmluZyBjYW4gYmUgZGVyaXZlZCB0d28gd2F5czogKDErMikrMyBvciAxKygyKzMpLlxuU2FtZSBraW5kIG9mIGlzc3Vl"
    "IHdpdGggY2hhaW5lZCA7IChzdGF0ZW1lbnRzKSBhbmQgY2hhaW5lZCDiiKcgKGJvb2xlYW5zKS4ifSwgIjUiOiB7ImEiOiB7Inki"
    "OiAyLCAieiI6IDV9LCAiZSI6ICJ4IHdhcyAxLCB1cGRhdGVkIHRvIDAgKHNvIGl0IGRyb3BzIG91dCBvZiB0aGUgY2Fub25pY2Fs"
    "IGZvcm0pLlxueSB3YXMgMiAodW50b3VjaGVkLCBrZXB0KS5cbnogZ2V0cyA1IChuZXcgZW50cnkpLiJ9LCAiNiI6IHsiYSI6IDE4"
    "LCAiZSI6ICJTdWJzdGl0dXRlIHggPSA0LCB0aGVuIGV2YWx1YXRlOiBBKCh4ICsgMikgKiAzLCDPgykgPSAoz4N4ICsgMikgKiAz"
    "LiJ9LCAiNyI6IHsiYSI6IHRydWUsICJlIjogIkV2YWx1YXRlIGVhY2ggY29uanVuY3Q6IHggPSAwIGluIM+DOyAhKHkgPD0gMykg"
    "aW4gz4M7IGNvbmpvaW4gd2l0aCDiiKcuIn0sICI4IjogeyJhIjogIkEiLCAiZSI6ICJUaGUgOj0gcnVsZTog4p+oeCA6PSBhLCDP"
    "g+KfqSDih5Ig4p+oc2tpcCwgz4NbeCDihqYgQSBhIM+DXeKfqS5cbkZvciBhIG51bWVyYWwgbiwgQShuLCDPgykgPSBuLiJ9LCAi"
    "OSI6IHsiYSI6ICJCIiwgImUiOiAiVGhlIHNraXAtOyBydWxlOiDin6hza2lwOyBULCDPg+KfqSDih5Ig4p+oVCwgz4Pin6kuXG5E"
    "cm9wIHRoZSBza2lwIGZyb20gdGhlIGxlZnQsIHN0YXRlIHVuY2hhbmdlZC4ifSwgIjEwIjogeyJhIjogIkIiLCAiZSI6ICJUaGUg"
    "aWYtZmYgcnVsZTogd2hlbiB0aGUgY29uZGl0aW9uIGlzIGZhbHNlLCB0YWtlIHRoZSBlbHNlLWJyYW5jaC5cblN0YXRlIHN0YXlz"
    "IM+DIOKAlCBjaGVja2luZyBhIGNvbmRpdGlvbiBkb2VzIG5vdCBtb2RpZnkgc3RhdGUuIn0sICIxMSI6IHsiYSI6ICJDIiwgImUi"
    "OiAiVGhlIHdoaWxlLXR0IHJ1bGUgdW5mb2xkcyB0aGUgbG9vcDogcnVuIGJvZHksIHRoZW4gcnVuIHRoZSBsb29wIGFnYWluLlxu"
    "VGhpcyBpcyB0aGUgcnVsZSB0aGF0IG1ha2VzIHdoaWxlLWxvb3BzIGFibGUgdG8gZXhwcmVzcyB1bmJvdW5kZWQgY29tcHV0YXRp"
    "b24g4oCUXG50aGUgcHJvZ3JhbSB0ZXh0IGdyb3dzIGR1cmluZyBhIGxvb3AgaXRlcmF0aW9uIGJlZm9yZSBpdCBzaHJpbmtzLiJ9"
    "LCAiMTIiOiB7ImEiOiAzLCAiZSI6ICJTdGVwIDE6IOKfqHg6PTE7IHk6PTIsIM+D4p+pIOKHkiDin6hza2lwOyB5Oj0yLCDPg1t4"
    "4oamMV3in6kgIHZpYSA7IHJ1bGVcblN0ZXAgMjog4oeSIOKfqHk6PTIsIM+DW3jihqYxXeKfqSAgdmlhIHNraXAtOyBydWxlXG5T"
    "dGVwIDM6IOKHkiDin6hza2lwLCDPg1t44oamMSwgeeKGpjJd4p+pICB2aWEgOj0gcnVsZVxuKFJlYWNoaW5nIHNraXAgZW5kcyB0"
    "aGUgdHJhY2Ug4oCUIHNraXAgaXRzZWxmIGlzIG5vdCBhIHRyYW5zaXRpb24uKSJ9LCAiMTMiOiB7ImEiOiBudWxsLCAiZSI6ICJU"
    "cmFjZSB4IHRocm91Z2ggZWFjaCBpdGVyYXRpb24uIFN0b3Agd2hlbiB0aGUgbG9vcCBjb25kaXRpb24gYmVjb21lcyBmYWxzZS4i"
    "fSwgIjE0IjogeyJhIjogZmFsc2UsICJlIjogIldhdGNoIHdoYXQgeCBkb2VzIGVhY2ggaXRlcmF0aW9uLiBEb2VzIGl0IGV2ZXIg"
    "c2F0aXNmeSB0aGUgbG9vcCBleGl0IGNvbmRpdGlvbj9cbldpdGggYXJiaXRyYXJ5LXByZWNpc2lvbiBpbnRlZ2VycywgdGhlcmUg"
    "aXMgbm8gdW5kZXJmbG93LiJ9LCAiMTUiOiB7ImEiOiAiQSIsICJlIjogIkV4ZXJjaXNlIDE3LiBQcm9vZiBieSBjYXNlIGFuYWx5"
    "c2lzIG9uIHdoaWNoIHJ1bGUgZmlyZXM6XG4gIEFzc2lnbm1lbnQgeSA6PSBhIG9ubHkgY2hhbmdlcyB5IChhbmQgeSDiiaAgeCBz"
    "aW5jZSBTIGhhcyBubyB4KS5cbiAgaWYvd2hpbGUgcnVsZXMgbGVhdmUgdGhlIHN0YXRlIHVuY2hhbmdlZC5cbiAgc2tpcC07IHJ1"
    "bGUgbGVhdmVzIHRoZSBzdGF0ZSB1bmNoYW5nZWQuXG5JbiBldmVyeSBjYXNlLCB0aGUgdmFsdWUgb2YgeCBpcyBwcmVzZXJ2ZWQu"
    "In0sICIxNiI6IHsiYSI6ICJCIiwgImUiOiAiU2VxdWVudGlhbCBjb21wb3NpdGlvbiBpcyBhc3NvY2lhdGl2ZSDigJQgaG93IHlv"
    "dSBwYXJlbnRoZXNpemUgY2hhaW5lZCA7IGRvZXMgbm90IGFmZmVjdFxudGVybWluYXRpb24gb3IgdGhlIGZpbmFsIHN0YXRlLiBU"
    "aGlzIGlzIHdoYXQgbWFrZXMgdGhlIEJORiBhbWJpZ3VpdHkgZm9yIGNoYWluZWQgO1xuaGFybWxlc3MuIE5vdGU6IDsgaXMgTk9U"
    "IGNvbW11dGF0aXZlIOKAlCBvcmRlciBhYnNvbHV0ZWx5IG1hdHRlcnMhIn0sICIxNyI6IHsiYSI6ICJCIiwgImUiOiAiRm9yIGFu"
    "eSAoUywgz4MpIHdpdGggUyDiiaAgc2tpcCwgZXhhY3RseSBvbmUgcnVsZSBhcHBsaWVzLCBhbmQgaXQgcHJvZHVjZXMgYSB1bmlx"
    "dWVseVxuZGV0ZXJtaW5lZCAoUycsIM+DJykuIFNvIHRoZSBzbWFsbC1zdGVwIHJlbGF0aW9uIGlzIGEgcGFydGlhbCBmdW5jdGlv"
    "biDigJQgbm9uLWRldGVybWluaXN0aWNcbmxhbmd1YWdlcyB3b3VsZCByZWxheCB0aGlzOyBXaGlsZSBkb2VzIG5vdC4ifSwgIjE4"
    "IjogeyJhIjogIkEiLCAiZSI6ICLin6hza2lwLCDPg+KfqSBpcyB0aGUgdGVybWluYWwgY29uZmlndXJhdGlvbiDigJQgbm8gcnVs"
    "ZSBhcHBsaWVzLCBzbyBubyBmdXJ0aGVyIHRyYW5zaXRpb25zLlxuQiBpcyB3cm9uZyAodmFyaWFibGVzIGNhbiBob2xkIGFueSBp"
    "bnQgZm9yZXZlcikuXG5DIGlzIHdyb25nIChjb3VsZCBiZSBhbnkgbnVtYmVyIG9mIHN0ZXBzOyBzb21lIHByb2dyYW1zIHJ1biBm"
    "b3JldmVyKS5cbkQgaXMgdG9vIG5hcnJvdyAoYSBwcm9ncmFtIGNhbiB0ZXJtaW5hdGUgd2l0aG91dCBhbnkgd2hpbGUgbG9vcHMg"
    "YXQgYWxsKS4ifSwgIjE5IjogeyJhIjogbnVsbCwgImUiOiAiV2FsayB0aGUgaWYtcnVsZSBmaXJzdCAod2hpY2ggYnJhbmNoIGZp"
    "cmVzPyksIHRoZW4gdGhlIGFzc2lnbm1lbnQuIn0sICIyMCI6IHsiYSI6IDMsICJlIjogIldhbGsgeCB0aHJvdWdoIGVhY2ggaXRl"
    "cmF0aW9uLiBTdG9wIGNvdW50aW5nIHdoZW4gdGhlIGxvb3AgZXhpdCBjb25kaXRpb24gaG9sZHMuIn19"
)
_KEY = json.loads(base64.b64decode(_KEY_B64).decode('utf-8'))


def _print_explain(explain: str) -> None:
    if explain:
        for line in explain.split('\n'):
            print(f'   {line}')


def check(qnum: int, predicted) -> None:
    """Look up the answer for question qnum, compare, record, report."""
    if predicted is Ellipsis:
        _RESULTS[qnum] = False
        print(f'Q{qnum}: ❌ Unanswered — replace ... with your answer.')
        return
    info = _KEY[str(qnum)]
    actual = info['a']
    correct = predicted == actual
    _RESULTS[qnum] = correct
    if correct:
        print(f'Q{qnum}: ✅ Correct.')
    else:
        print(f'Q{qnum}: ❌ Incorrect.')
        print(f'  You answered: {predicted!r}')
        print(f'  Correct:      {actual!r}')
    _print_explain(info['e'])


def check_state(qnum: int, predicted, prog: str, state: dict | None = None,
                max_steps: int = 10_000) -> None:
    """Run the program to compute the actual final state; compare; record; report."""
    if predicted is Ellipsis:
        _RESULTS[qnum] = False
        print(f'Q{qnum}: ❌ Unanswered — replace ... with a dict.')
        return
    actual = run(prog, state or {}, max_steps=max_steps)
    pred_canon = {k: v for k, v in predicted.items() if v != 0}
    correct = pred_canon == actual
    _RESULTS[qnum] = correct
    if correct:
        print(f'Q{qnum}: ✅ Correct. Final state: {actual}')
    else:
        print(f'Q{qnum}: ❌ Incorrect.')
        print(f'  You answered: {pred_canon!r}')
        print(f'  Correct:      {actual!r}')
    _print_explain(_KEY[str(qnum)]['e'])


print('quiz harness ready')

quiz harness ready


## Section A — Syntax & Grammar

### Q1 — Valid arithmetic expression

Which of these is a valid `<aexp>` per the BNF grammar of While?

- **A.** `tt`
- **B.** `x + 1`
- **C.** `x = 0`
- **D.** `if 1 then 2 else 3`

In [2]:
q1 = ...   # 'A', 'B', 'C', or 'D'
check(1, q1)

Q1: ❌ Unanswered — replace ... with your answer.


### Q2 — NOT a valid boolean expression

Which of these is **NOT** a valid `<bexp>`?

- **A.** `x = 1`
- **B.** `tt`
- **C.** `x + 1`
- **D.** `!(x <= y)`

In [3]:
q2 = ...   # 'A', 'B', 'C', or 'D'
check(2, q2)

Q2: ❌ Unanswered — replace ... with your answer.


### Q3 — Encode disjunction `a ∨ b`

Which is the correct encoding of `a ∨ b` in our ASCII syntax (where `a` and `b` are arbitrary boolean expressions)?

- **A.** `a | b`
- **B.** `a + b`
- **C.** `!((!a) & (!b))`
- **D.** `!(a & b)`

In [4]:
q3 = ...
check(3, q3)

Q3: ❌ Unanswered — replace ... with your answer.


### Q4 — Why is the BNF grammar ambiguous?

- **A.** Because it has too many statement types
- **B.** Because rules like `<aexp> ::= <aexp> + <aexp>` allow multiple parse trees for `1 + 2 + 3`
- **C.** Because it allows infinite recursion
- **D.** Because it doesn't define if-statements

In [5]:
q4 = ...
check(4, q4)

Q4: ❌ Unanswered — replace ... with your answer.


## Section B — States and updates

### Q5 — State after multi-update

Compute the resulting state:

$$\{x \mapsto 1, y \mapsto 2\}[x \mapsto 0, z \mapsto 5]$$

Fill in the canonical form (omit zero-valued variables).

In [6]:
# Type the resulting state as a Python dict — only non-zero entries.
q5 = ...   # e.g. {'a': 1, 'b': 2}
check(5, q5)

Q5: ❌ Unanswered — replace ... with your answer.


## Section C — Evaluators A and B

### Q6 — Evaluate an arithmetic expression

What is `A((x + 2) * 3, {x: 4})`?

In [7]:
q6 = ...   # an integer
check(6, q6)

Q6: ❌ Unanswered — replace ... with your answer.


### Q7 — Evaluate a boolean expression

What is `B((x = 0) & !(y <= 3), {x: 0, y: 5})`?

In [8]:
q7 = ...   # True or False
check(7, q7)

Q7: ❌ Unanswered — replace ... with your answer.


## Section D — Small-step rules

### Q8 — Assignment rule

Given `⟨x := 7, σ⟩`, the next configuration is:

- **A.** `⟨skip, σ[x ↦ 7]⟩`
- **B.** `⟨skip, σ⟩`
- **C.** `⟨x := 7, σ[x ↦ 7]⟩`
- **D.** `⟨skip, σ[x ↦ 0]⟩`

In [9]:
q8 = ...
check(8, q8)

Q8: ❌ Unanswered — replace ... with your answer.


### Q9 — `skip; T` rule

Given `⟨skip; T, σ⟩`, the next configuration is:

- **A.** `⟨skip, σ⟩`
- **B.** `⟨T, σ⟩`
- **C.** `⟨skip; T, σ⟩`
- **D.** `⟨T; skip, σ⟩`

In [10]:
q9 = ...
check(9, q9)

Q9: ❌ Unanswered — replace ... with your answer.


### Q10 — `if` with false condition

Given `⟨if b then S else (S'), σ⟩` with `B(b, σ) = ff`, the next configuration is:

- **A.** `⟨S, σ⟩`
- **B.** `⟨S', σ⟩`
- **C.** `⟨skip, σ⟩`
- **D.** `⟨S; S', σ⟩`

In [11]:
q10 = ...
check(10, q10)

Q10: ❌ Unanswered — replace ... with your answer.


### Q11 — `while` with true condition

Given `⟨while b do (S), σ⟩` with `B(b, σ) = tt`, the next configuration is:

- **A.** `⟨skip, σ⟩`
- **B.** `⟨S, σ⟩`
- **C.** `⟨S; while b do (S), σ⟩`
- **D.** `⟨while b do (S), σ⟩`

In [12]:
q11 = ...
check(11, q11)

Q11: ❌ Unanswered — replace ... with your answer.


## Section E — Tracing programs

### Q12 — Count transitions of a tiny program

How many small-step transitions does the program `x := 1; y := 2` take starting from `σ = {}`? Count every `⇒` arrow until you reach `⟨skip, σ'⟩`.

In [13]:
q12 = ...   # an integer
check(12, q12)

Q12: ❌ Unanswered — replace ... with your answer.


### Q13 — Final state of a small loop

What's the final state of

```
x := 0;
while x <= 2 do (x := x + 1)
```

starting from `σ = {}`? Give the canonical form (no zero-valued variables).

In [14]:
q13 = ...   # a dict
check_state(13, q13, 'x := 0; while x <= 2 do (x := x + 1)')

Q13: ❌ Unanswered — replace ... with a dict.


### Q14 — Termination check

Does this program terminate from `σ = {}`?

```
x := 0;
while !(x = 5) do (x := x - 1)
```

In [15]:
q14 = ...   # True (terminates) or False (doesn't)
check(14, q14)

Q14: ❌ Unanswered — replace ... with your answer.


## Section F — Reasoning about the semantics

### Q15 — State preservation (Exercise 17)

If `⟨S, σ⟩ ⇒ ⟨S', σ'⟩` and `S` does **not** mention the variable `x`, then:

- **A.** `σ'(x) = σ(x)` — x is preserved
- **B.** `σ'(x) = 0`
- **C.** `σ'(x)` is undefined
- **D.** `σ'(x)` can be anything

In [16]:
q15 = ...
check(15, q15)

Q15: ❌ Unanswered — replace ... with your answer.


### Q16 — Associativity of `;` (Proposition 3)

What does Proposition 3 establish?

- **A.** Every While program terminates
- **B.** `⟨(S; T); U, σ⟩ ⇒* ⟨skip, τ⟩` iff `⟨S; (T; U), σ⟩ ⇒* ⟨skip, τ⟩`
- **C.** `;` is commutative — the order of statements doesn't matter
- **D.** `skip` is the identity element for `;`

In [17]:
q16 = ...
check(16, q16)

Q16: ❌ Unanswered — replace ... with your answer.


### Q17 — Determinism (Exercise 18)

If `⟨S, σ⟩ ⇒ ⟨S', ρ⟩` and `⟨S, σ⟩ ⇒ ⟨S', τ⟩` then `ρ = τ`. This implies:

- **A.** While is non-deterministic — multiple next states are possible
- **B.** While is **deterministic** — each (S, σ) has at most one next configuration
- **C.** `skip` is the unique terminal program
- **D.** Variables can change in unpredictable ways

In [18]:
q17 = ...
check(17, q17)

Q17: ❌ Unanswered — replace ... with your answer.


### Q18 — When does a program terminate?

A While program terminates when:

- **A.** The configuration reaches `⟨skip, σ⟩` for some σ
- **B.** All variables become 0
- **C.** After exactly n steps for some n
- **D.** The boolean condition of the outermost loop becomes false

In [19]:
q18 = ...
check(18, q18)

Q18: ❌ Unanswered — replace ... with your answer.


### Q19 — Mixed program with if + assignment

What's the final state of

```
if x = 0 then
    y := 1
else (
    y := 2
);
z := y + 10
```

starting from `σ = {x: 5}`? Canonical form.

In [20]:
q19 = ...   # a dict
check_state(19, q19,
            'if x = 0 then (y := 1) else (y := 2); z := y + 10',
            state={'x': 5})

Q19: ❌ Unanswered — replace ... with a dict.


### Q20 — Iteration count

How many times does the **body** of this loop execute, starting from `σ = {x: 0}`?

```
while x <= 4 do (x := x + 2)
```

In [21]:
q20 = ...   # an integer
check(20, q20)

Q20: ❌ Unanswered — replace ... with your answer.


## Final score

Run this cell once you've answered all 20 questions.

In [22]:
total = len(_RESULTS)
correct = sum(_RESULTS.values())
missing = [n for n in range(1, 21) if n not in _RESULTS]

print(f'Score: {correct}/{total}')
if missing:
    print(f'  (you have not answered: {missing})')

if total == 0:
    print('No answers recorded — go run the question cells.')
elif correct == 20:
    print('🎉 Perfect — every concept covered. You are ready.')
elif correct >= 18:
    print('Strong grasp. Spot-check the questions you missed and you are exam-ready.')
elif correct >= 15:
    print('Solid. Re-read the section(s) the missed questions came from.')
elif correct >= 12:
    print('Mid-pack. Worth one more pass through N3 (semantics) before the exam.')
else:
    print('Worth a full re-read. Start again at N1, slow down on predict-cells.')

print()
print('Question-by-question:')
section_map = {
    'A — Syntax & Grammar':            range(1, 5),
    'B — States and updates':          range(5, 6),
    'C — Evaluators A and B':          range(6, 8),
    'D — Small-step rules':            range(8, 12),
    'E — Tracing programs':            list(range(12, 15)) + [19, 20],
    'F — Reasoning about the semantics': range(15, 19),
}
for section, qnums in section_map.items():
    qnums = list(qnums)
    sec_correct = sum(_RESULTS.get(q, False) for q in qnums)
    sec_total = len(qnums)
    marks = ' '.join('✅' if _RESULTS.get(q) else ('❌' if q in _RESULTS else '·') for q in qnums)
    print(f'  {section}: {sec_correct}/{sec_total}   {marks}')

Score: 0/20
Worth a full re-read. Start again at N1, slow down on predict-cells.

Question-by-question:
  A — Syntax & Grammar: 0/4   ❌ ❌ ❌ ❌
  B — States and updates: 0/1   ❌
  C — Evaluators A and B: 0/2   ❌ ❌
  D — Small-step rules: 0/4   ❌ ❌ ❌ ❌
  E — Tracing programs: 0/5   ❌ ❌ ❌ ❌ ❌
  F — Reasoning about the semantics: 0/4   ❌ ❌ ❌ ❌


## Where to go from here

If you missed questions in:

- **Section A (syntax)** → re-read N2 §1–4. The BNF rules are the foundation.
- **Section B (states)** → re-read N3 §1–2 plus Exercise 11.
- **Section C (A and B)** → re-read N3 §3 — the truth tables are short, and they recur in every question that involves an if or while.
- **Section D (rules)** → this is the heart of the material. Re-read N3 §4 carefully. Try producing the formal trace for Example 1 from scratch on paper.
- **Section E (tracing)** → run more programs through `trace(prog, σ, view='formal')`. Predict each step before the interpreter shows you.
- **Section F (reasoning)** → re-read N4 §1–3. The proofs are by case-analysis on rules; once that pattern clicks, the answers feel mechanical.